# 4.0 — CL Setup (run this first)

> Design: `PIPELINE.md`. Needs: Section 2 cache + Section 3 decision.

Builds the **base head** on Task 0 and freezes the protocol every arm will share.

## Why a shared setup notebook

Each method now lives in its own notebook (4.1–4.6). For the comparison to mean
anything they must all start from the **same weights** and use the **same**
lr / epochs / batch / schedule. This notebook writes both:

```
head_base.pt            <- identical starting point for every arm
stage1_protocol.json    <- identical hyperparameters for every arm
```

Every arm notebook loads these. The only thing that differs between them is which
mechanism flag is switched on.

## The arms

| Notebook | Method | Old-class signal | Memory/class |
|---|---|---|---|
| 4.1 | Naive | none | 0 |
| 4.2 | WeightAlign | logit rescale | 0 |
| 4.3 | EWC | weight anchor | Fisher + theta* |
| 4.4 | **LwF** | **soft targets, every sample** | **0** |
| 4.5 | SmartReplay | hard labels, 15 exemplars | ~960 KB |
| 4.6 | **Replay + LwF** (iCaRL) | **both** | ~960 KB |
| 4.7 | Compare | — | — |

Run 4.0 once, then 4.1–4.6 in any order, then 4.7.

## Setup

In [ ]:
# ── ULAL bootstrap — self-contained, no Drive dependency ────────────────────
# One shallow git clone instead of hundreds of API calls. subprocess takes a
# LIST, never a shell string, so an '&' in a branch name is safe.
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden): ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

# ── Verify the clone contains what this notebook needs ──────────────────────
# The clone reflects what is ON GITHUB. Local commits you never pushed are
# invisible here, and the symptom is a confusing ModuleNotFoundError later.
_need = ["src/__init__.py", "src/bootstrap.py", "src/imports.py",
         "src/data/cache.py", "src/cl/trainer.py",
         "src/models/temporal_head.py", "src/utils/probe.py"]
_miss = [f for f in _need if not os.path.exists(os.path.join(ROOT, f))]

if _miss:
    print("ERROR — the cloned repo is missing:")
    for f in _miss:
        print("   ", f)
    print("")
    print("These exist locally but are not on the '" + BRANCH + "' branch on GitHub.")
    print("On your machine run:")
    print("    git add -A")
    print("    git commit -m 'sync src'")
    print("    git push origin " + BRANCH)
    print("then re-run this cell.")
    raise RuntimeError("src/ incomplete on GitHub — push your commits first.")

print("src/ complete (" + str(len(_need)) + " key files present)")

In [ ]:
# ── Drive + dataset + paths ─────────────────────────────────────────────────
# Exports ULAL_* env vars that config.py reads. Nothing is written to disk and
# no kernel restart is needed.
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
# ── Global label space (src/data/cache.py) ──────────────────────────────────
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]     # Sections 4-5 stream over these
FEWSHOT_TASK = 3          # Section 7 probe only

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

In [ ]:
# ── Load cached frame features (src/data/cache.py) ──────────────────────────
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

## Read Section 3's decision

In [ ]:
_p = f"{cfg.RESULTS_DIR}/stage0_freeze_study.json"
if os.path.exists(_p):
    _fs = json.load(open(_p))
    FREEZE_LSTM, CEILING = _fs["chosen_freeze_lstm"], _fs["ceiling"]
    print(f"From Section 3: FREEZE_LSTM={FREEZE_LSTM}  CEILING={CEILING:.2%}")
else:
    FREEZE_LSTM, CEILING = False, None
    print("WARNING: no stage0_freeze_study.json — run Section 3 first.")

## Train the base head on Task 0

In [ ]:
CL_EPOCHS, CL_LR, CL_WD, CL_BATCH, CL_LS = 15, 5e-4, 0.03, 32, 0.10

base_s, base_y = task_split(CACHE, 0, "train")
val0_s, val0_y = task_split(CACHE, 0, "val")

base_head = make_temporal_head(NUM_AFTER[0], freeze_lstm=FREEZE_LSTM,
                               lstm_state=LSTM_STATE, device=device)
base_head = finetune_head(
    base_head, base_s, None, base_y, val0_s, val0_y, device,   # train_t=None -> no MSE
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH,
    patience=cfg.FINETUNE_PATIENCE,
    ce_weight=1.0, mse_weight=0.0, label_smoothing=CL_LS,
)

acc0 = eval_tasks_upto(base_head, CACHE, 0, device)
print(f"\nBase head — TEST accuracy: {acc0[0]:.2%}")
if CEILING:
    print(f"(ceiling for all {NUM_AFTER[-1]} classes was {CEILING:.2%})")

## Save head + protocol

In [ ]:
os.makedirs(cfg.CKPT_DIR, exist_ok=True)
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
torch.save(base_head.state_dict(), f"{cfg.CKPT_DIR}/head_base.pt")

protocol = {
    "freeze_lstm": bool(FREEZE_LSTM), "ceiling": CEILING,
    "epochs": CL_EPOCHS, "lr": CL_LR, "wd": CL_WD,
    "batch_size": CL_BATCH, "label_smoothing": CL_LS,
    "seed": cfg.SEED,
    "num_after": [int(v) for v in NUM_AFTER],
    "task_names": TASK_NAMES, "cl_tasks": CL_TASKS,
    "global_classes": global_classes,
    "base_acc": acc0[0],
    "note": "fixed epochs, no best-val selection — identical for every arm",
}
with open(f"{cfg.RESULTS_DIR}/stage1_protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

print(json.dumps(protocol, indent=2)[:600] + " ...")
print(f"\nSaved -> {cfg.CKPT_DIR}/head_base.pt")
print(f"Saved -> {cfg.RESULTS_DIR}/stage1_protocol.json")
print("\nNow run 4.1 - 4.6 (any order), then 4.7 to compare.")